Predecir resultados en partidas multijugador
🧠 Objetivo

En este ejercicio, aplicarás tus conocimientos de regresión logística para construir un modelo capaz de predecir si un jugador ganó o perdió una partida, a partir de sus estadísticas individuales.



📋 Descripción del problema

Tienes que construir un modelo predictivo que, a partir de las estadísticas de un jugador en una partida, determine si ganó o no.

Para ello, deberás:

Crear datos sintéticos que representen partidas ficticias de jugadores.

Entrenar un modelo de regresión logística con esos datos.

Implementar una función que prediga el resultado (ganar o no) para un nuevo jugador.



📦 Paso 1: Definir una clase para representar una partida

Crea una clase PlayerMatchData con los siguientes atributos:

kills: número de enemigos eliminados

deaths: número de veces que el jugador ha muerto

assists: asistencias realizadas

damage_dealt: daño total infligido

damage_received: daño total recibido

healing_done: curación realizada

objective_time: tiempo (en segundos) que el jugador estuvo capturando objetivos

won: 1 si el jugador ganó la partida, 0 si perdió

Incluye un método .to_dict() que devuelva los datos como un diccionario (sin la variable won, opcionalmente).



📦 Paso 2: Generar datos sintéticos con NumPy

Crea una función llamada generate_synthetic_data que genere un conjunto de datos de entrenamiento simulando partidas de videojuegos. Para ello:

Utiliza la librería numpy para generar los valores numéricos.

Cada instancia representará el desempeño de un jugador en una partida.

La función debe devolver una lista de objetos PlayerMatchData (ya definida previamente).

Implementa la siguiente lógica para cada jugador:

Reglas para los datos:

kills: número de enemigos eliminados, generado con una distribución de Poisson con media 5.

kills = np.random.poisson(5)
deaths: número de veces que el jugador ha muerto, distribución de Poisson con media 3.

assists: asistencias realizadas, distribución de Poisson con media 2.

damage_dealt: daño infligido, calculado como kills * 300 + ruido aleatorio normal.

damage_received = deaths * 400 + np.random.normal(0, 100)
damage_received: daño recibido, como deaths * 400 + ruido aleatorio normal.

healing_done: cantidad de curación, valor aleatorio entero entre 0 y 300.

objective_time: tiempo (en segundos) controlando objetivos, valor aleatorio entre 0 y 120.

won: el jugador se considera que ganó la partida si hizo más daño del que recibió y tuvo más kills que muertes.

🧠 Tu función debe seguir esta estructura:

import numpy as np
 
def generate_synthetic_data(n=100):
    data = []
    for _ in range(n):
        # Genera cada variable siguiendo las instrucciones dadas
        # Crea un objeto PlayerMatchData con estos valores
        # Añádelo a la lista de datos
 
    return data


🧪 Paso 3: Crear y entrenar el modelo

Crea una clase VictoryPredictor que entrene un modelo de regresión logística con los datos sintéticos. Esta clase debe tener:

Un método train(data) para entrenar el modelo.

Un método predict(player: PlayerMatchData) que devuelva 1 si predice victoria, 0 si derrota.



📌 Ejemplo de uso

### Crear datos de entrenamiento
training_data = generate_synthetic_data(150)
 
### Entrenar modelo
predictor = VictoryPredictor()
predictor.train(training_data)
 
### Crear jugador de prueba
test_player = PlayerMatchData(8, 2, 3, 2400, 800, 120, 90, None)
 
### Predecir si ganará
prediction = predictor.predict(test_player)
print(f"¿El jugador ganará? {'Sí' if prediction == 1 else 'No'}")


Salida esperada

¿El jugador ganará? Sí


Practicar con numpy para genera datos sintéticos y con algoritmos de regresión logística para realizar predicciones


# Solucion:

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from typing import List, Optional

class PlayerMatchData:
    def __init__(self, kills: int, deaths: int, assists: int,
                 damage_dealt: float, damage_received: float,
                 healing_done: float, objective_time: int,
                 won: Optional[int]):
        
        self.kills = kills
        self.deaths = deaths
        self.assists = assists
        self.damage_dealt = damage_dealt
        self.damage_received = damage_received
        self.healing_done = healing_done
        self.objective_time = objective_time
        self.won = won
        
    def to_dict(self) -> dict:
        return {
            "kills": self.kills,
            "deaths": self.deaths,
            "assists": self.assists,
            "damage_dealt": self.damage_dealt,
            "damage_received": self.damage_received,
            "healing_done": self.healing_done,
            "objective_time": self.objective_time
        }
        
    def __repr__(self) -> str:
        return f"PlayerMatchData(KDA={self.kills}/{self.deaths}/{self.assists}, Won={self.won})"

def generate_synthetic_data(n: int = 100) -> List[PlayerMatchData]:
    data = []
    for _ in range(n):
        kills = np.random.poisson(5)
        deaths = np.random.poisson(3)
        assists = np.random.poisson(2)
        damage_dealt = (kills * 300) + np.random.normal(0, 75)
        damage_received = (deaths * 400) + np.random.normal(0, 100)
        healing_done = np.random.randint(0, 301)
        objective_time = np.random.randint(0, 121)
        
        won = 0
        
        
        if (damage_dealt > damage_received) and (kills > deaths):
            won = 1
        
        match = PlayerMatchData(
            kills=kills,
            deaths=deaths,
            assists=assists,
            damage_dealt=damage_dealt,
            damage_received=damage_received,
            healing_done=healing_done,
            objective_time=objective_time,
            won=won
        )
        
        data.append(match)
    
    return data # Este es el único 'return' de la función
class VictoryPredictor: 
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = LogisticRegression(random_state=42)
        self.feature_columns = []
        
    def train(self, data: List[PlayerMatchData]):
        X_data = [d.to_dict() for d in data]
        X = pd.DataFrame(X_data)
        self.feature_columns = list(X.columns)
        y = [d.won for d in data]
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        
    def predict(self, player: PlayerMatchData) -> int:
        player_dict = player.to_dict()
        player_df = pd.DataFrame([player_dict])
        player_df = player_df[self.feature_columns]
        player_scaled = self.scaler.transform(player_df)
        prediction = self.model.predict(player_scaled)
        return prediction[0]
        

training_data = generate_synthetic_data(500)
predictor = VictoryPredictor()
predictor.train(training_data)

test_player = PlayerMatchData(
    kills=8,
    deaths=2,
    assists=3,
    damage_dealt=2400,
    damage_received=800,
    healing_done=120,
    objective_time=90,
    won=None
)
prediction = predictor.predict(test_player)
print(f"Jugador de prueba 1 (8K/2D): ¿El jugador ganará? {'Sí' if prediction == 1 else 'No'}")

test_player_2 = PlayerMatchData(
    kills=2, 
    deaths=10, 
    assists=1, 
    damage_dealt=600, 
    damage_received=4000, 
    healing_done=50, 
    objective_time=10, 
    won=None
)
prediction_2 = predictor.predict(test_player_2)
print(f"Jugador de prueba 2 (2K/10D): ¿El jugador ganará? {'Sí' if prediction_2 == 1 else 'No'}")

Jugador de prueba 1 (8K/2D): ¿El jugador ganará? Sí
Jugador de prueba 2 (2K/10D): ¿El jugador ganará? No


Casos de prueba

Suspenso: 0, Aprobado: 2 de 2 pruebas

test_predict_consistency

test_predict_output_type
